In [1]:
import numpy as np
import csv
import pandas as pd
from itertools import islice
import json
import missingno as msno
import matplotlib.pyplot as plt
import sklearn

In [2]:
books = pd.read_csv('../../data/data_with_author_and_awards.csv')

/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_18017/475203437.py:1: DtypeWarning: Columns (0: isbn) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('../../data/data_with_author_and_awards.csv')


In [3]:
books.head(5)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
0,9372.0,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914.0,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,...,False,False,Unknown,40,0,0,False,False,USA,Central/North America
1,2215181.0,Mrs. Candy Strikes It Rich,Robert Tallant,1954,1954-00-00,Doubleday,1909.0,"New Orleans, Louisiana, USA",NaN,NaN,...,False,False,Unknown,45,0,0,False,False,USA,Central/North America
2,1392436.0,Return to the Lost Planet,Angus MacVicar,1954,1954-00-00,Burke,1908.0,"Duror, Argyll, Scotland, UK",NaN,NaN,...,False,False,Unknown,46,0,0,False,False,UK,Europe
3,1112206.0,Rainbow on the Road,Esther Forbes,1954,1954-00-00,Houghton Mifflin,1891.0,"Westborough, Massachusetts, USA",NaN,NaN,...,False,False,Unknown,63,0,0,False,False,USA,Central/North America
4,1908.0,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896.0,"Norfolk, Virginia, USA",0881846163,"**From the first page of the Ace Double:** ""Na...",...,False,False,Unknown,58,0,0,False,False,USA,Central/North America


In [4]:
# apply data cleaning stuff 

%run ../../data/tidy_book_tags.ipynb # to get clean_filter_tags function

%run ../../data/tidy_publisher.ipynb # to get publisher_tidy function


In [5]:
books['first_publisher'] = publisher_tidy(books['first_publisher'])

books['tags'] = clean_filter_tags(books['tags'])

In [6]:
# combine awards into one target column

books['target'] = books['hugo'] | books['locus']

try some ML model

In [7]:
0.2*(2025 - 1971)

10.8

In [33]:
books = books.dropna()

books_tt = books[(books['release_year'] >= 1971) & (books['release_year'] < 2010)]
books_val = books[(books['release_year'] >= 2010) & (books['release_year'] < 2015)]

books_test = books[(books['release_year'] >= 2015)]

In [34]:
books_tt['Author_Age_at_Publication'] = pd.to_numeric(
    books_tt['Author_Age_at_Publication'], errors='coerce'
)


In [35]:
books.columns

Index(['title_id', 'title', 'author', 'release_year', 'release_date',
       'first_publisher', 'author_birthyear', 'author_birthplace', 'isbn',
       'book_synopsis', 'tags', 'hugo', 'locus', 'month_of_publication',
       'Author_Age_at_Publication', 'Hugo_Awards_Previously',
       'Locus_Awards_Previously', 'Hugo_Nominee_Before',
       'Locus_Nominee_Before', 'author_birthplace_country',
       'author_birthplace_continent', 'target'],
      dtype='str')

In [36]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

# scale = StandardScaler()
categorize = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

mlb = MultiLabelBinarizer()
tags_binarized = mlb.fit_transform(books_tt['tags'].fillna(''))
tags_df = pd.DataFrame(
    tags_binarized, 
    columns=[f"tag_{c}" for c in mlb.classes_],
    index=books_tt.index
)
books_tt = pd.concat([books_tt, tags_df], axis=1)

numerical_features = ['Author_Age_at_Publication', 'Hugo_Awards_Previously', 'Locus_Awards_Previously', 'Hugo_Nominee_Before',  'Locus_Nominee_Before']
cat_features = ['first_publisher']
mlb_features = ['tags']

preprocessing = ColumnTransformer(
    transformers=[
        # ('numerical', SimpleImputer(strategy='median'), numerical_features),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('passthrough', 'passthrough', numerical_features)
        # ('multilabel', mlb, mlb_features)
    ],
    remainder='passthrough'
)

features = numerical_features + cat_features + list(tags_df.columns) 

models = {
    'dummy_mostfrequent': Pipeline(steps=[
        ('preprocess', preprocessing),
        ('modelling', DummyClassifier(strategy='most_frequent'))
    ]
    ),
    'dummy_stratified': Pipeline(steps=[
        ('preprocess', preprocessing),
        ('modelling', DummyClassifier(strategy='stratified'))
    ]
    ),
    'simple_classifier': Pipeline(steps=[
        ('preprocess', preprocessing),
        ('logistic', LogisticRegression(max_iter=1000))
    ]
    ),
    'simple_classifier_balanced': Pipeline(steps=[
        ('preprocess', preprocessing),
        ('logistic', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]
    )
}

In [37]:
for name, model in models.items():
    print(name)
    model.fit(books_tt[features], books_tt['target'])

dummy_mostfrequent
dummy_stratified
simple_classifier
simple_classifier_balanced


In [38]:
# evaluate on validation set quickly:

tags_binarized = mlb.transform(books_val['tags'].fillna(''))
tags_df = pd.DataFrame(
tags_binarized, 
columns=[f"tag_{c}" for c in mlb.classes_],
index=books_val.index
)
books_val = pd.concat([books_val, tags_df], axis=1)

scores = {}
preds = {}

for name, model in models.items():

    ypred = model.predict(books_val[features])

    scores[name]= f1_score(books_val['target'], ypred)
    preds[name] = ypred


/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['standalon'] will be ignored
  warnings.warn(


In [39]:
scores

{'dummy_mostfrequent': 0.0,
 'dummy_stratified': 0.042826552462526764,
 'simple_classifier': 0.5085910652920962,
 'simple_classifier_balanced': 0.5423728813559322}

some results:
- validation results on just author info + publisher: 0.41
- validation results on author info + publisher + tags: 0.50

In [40]:
confusion_matrix(books_val['target'], preds['simple_classifier'], normalize='true')

array([[0.98350742, 0.01649258],
       [0.52866242, 0.47133758]])

In [42]:
confusion_matrix(books_val['target'], preds['simple_classifier_balanced'], normalize='true')

array([[0.93677845, 0.06322155],
       [0.08280255, 0.91719745]])

this one bad model is, expectedly, pretty bad at telling between nominees and non-nominees. 
we should try to do some more preprocessing and normalizing against cohort etc to try and do better.